In [ ]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
import os

parent_dir = Path.cwd().parent.resolve() # move one level up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

databases = ['bank_notes', 'forage', 'milk', 'soil', 'soil_types', 'synthetic']
models = ['PLS', 'SVM', 'MLP']

# Carregar os valores de rbo de acordo com cada database e modelo
rbo_results = {}
rbo_pca_results = {}
files_found = []
files_not_found = []

# Carregar dados dos modelos base
for db in databases:
    rbo_results[db] = {}
    for model in models:
        file_path = parent_dir / model / db / 'rbo_rank.csv'
        
        if file_path.exists():
            try:
                rbo_results[db][model] = pd.read_csv(file_path, sep=';')
                files_found.append(f'{model}/{db}/rbo_rank.csv')
            except Exception as e:
                print(f"Erro ao ler {file_path}: {e}")
                files_not_found.append(f'{model}/{db}/rbo_rank.csv (erro: {e})')
                rbo_results[db][model] = None
        else:
            files_not_found.append(f'{model}/{db}/rbo_rank.csv (não existe)')
            rbo_results[db][model] = None

# Carregar dados do PCA_aggregator
for db in databases:
    rbo_pca_results[db] = {}
    for model in models:
        file_path = parent_dir / 'PCA_aggregator' / model / db / 'rbo_rank.csv'
        
        if file_path.exists():
            try:
                rbo_pca_results[db][model] = pd.read_csv(file_path, sep=';')
                files_found.append(f'PCA_aggregator/{model}/{db}/rbo_rank.csv')
            except Exception as e:
                print(f"Erro ao ler {file_path}: {e}")
                files_not_found.append(f'PCA_aggregator/{model}/{db}/rbo_rank.csv (erro: {e})')
                rbo_pca_results[db][model] = None
        else:
            files_not_found.append(f'PCA_aggregator/{model}/{db}/rbo_rank.csv (não existe)')
            rbo_pca_results[db][model] = None

print(f"\n✓ Arquivos encontrados e carregados: {len(files_found)}")
print(f"  - Modelos base: {sum(1 for f in files_found if not f.startswith('PCA'))}")
print(f"  - PCA_aggregator: {sum(1 for f in files_found if f.startswith('PCA'))}")
print(f"✗ Arquivos não encontrados: {len(files_not_found)}")

if files_not_found:
    print("\nArquivos ausentes:")
    for f in files_not_found[:10]:  # Mostrar apenas os primeiros 10
        print(f"  - {f}")
    if len(files_not_found) > 10:
        print(f"  ... e mais {len(files_not_found) - 10} arquivo(s)")

In [ ]:
# Juntar os arquivos de rbo em um único dataframe para forage

rbo_forage_list = []

# Adicionar dados dos modelos base
for model_name, df in rbo_results['forage'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = model_name
        rbo_forage_list.append(df_copy)

# Adicionar dados do PCA_aggregator
for model_name, df in rbo_pca_results['forage'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = f'{model_name}_PCA'
        rbo_forage_list.append(df_copy)

rbo_forage = pd.concat(rbo_forage_list, ignore_index=True)

print(f"Total de linhas: {len(rbo_forage)}")
print(f"Modelos: {sorted(rbo_forage['Model'].unique().tolist())}")
rbo_forage

In [ ]:
# Criar matrizes de comparação para cada modelo (formato co-ocorrência)
# Method_1 nas linhas, Method_2 nas colunas
# Matriz triangular inferior preenchida (abaixo da diagonal)

print("="*80)
print("CRIANDO MATRIZES DE COMPARAÇÃO POR MODELO (FORAGE)")
print("="*80)

# Dicionário para armazenar as matrizes de cada modelo
forage_matrices = {}

# Obter lista de modelos únicos
models_list = sorted(rbo_forage['Model'].unique())

print(f"\nModelos encontrados: {models_list}\n")

# Para cada modelo, criar uma matriz pivotada triangular inferior
for model in models_list:
    # Filtrar dados do modelo
    model_data = rbo_forage[rbo_forage['Model'] == model].copy()
    
    # Identificar a coluna de valores (deve ser numérica e não Method_1/Method_2/Model)
    value_cols = [col for col in model_data.columns 
                  if col not in ['Method_1', 'Method_2', 'Model'] 
                  and pd.api.types.is_numeric_dtype(model_data[col])]
    
    if value_cols:
        value_col = value_cols[0]  # Usar a primeira coluna de valores encontrada
        
        # Reorganizar para manter apenas triangular inferior
        # Se existir (A, B) e (B, A), combinar mantendo apenas onde row >= col
        model_data_processed = []
        
        for _, row in model_data.iterrows():
            m1, m2, val = row['Method_1'], row['Method_2'], row[value_col]
            
            # Sempre colocar o "maior" método na linha (Method_1)
            if m1 < m2:  # Comparação lexicográfica
                m1, m2 = m2, m1  # Trocar
            
            model_data_processed.append({
                'Method_1': m1,
                'Method_2': m2,
                value_col: val
            })
        
        # Criar DataFrame processado
        df_processed = pd.DataFrame(model_data_processed)
        
        # Remover duplicatas (manter apenas uma entrada por par)
        df_processed = df_processed.drop_duplicates(subset=['Method_1', 'Method_2'], keep='first')
        
        # Criar matriz pivot
        matrix = df_processed.pivot(
            index='Method_1',
            columns='Method_2',
            values=value_col
        )
        
        # Garantir que a ordem seja consistente (ordenar índice e colunas)
        matrix = matrix.sort_index(axis=0).sort_index(axis=1)
        
        forage_matrices[model] = matrix
        
        print(f"✅ {model.ljust(15)} → Matriz {matrix.shape[0]}x{matrix.shape[1]} ({value_col}) - Triangular inferior")
    else:
        print(f"⚠️  {model.ljust(15)} → Nenhuma coluna de valores encontrada")

print(f"\n{'='*80}")
print(f"✅ Total: {len(forage_matrices)} matrizes criadas")
print(f"{'='*80}")

# Exibir exemplo de uma matriz
if forage_matrices:
    example_model = models_list[0]
    print(f"\n📊 EXEMPLO - Matriz triangular inferior para {example_model}:")
    print(forage_matrices[example_model])
    print("\n💡 Valores abaixo da diagonal principal estão preenchidos, acima estão vazios (NaN)")


In [ ]:
forage_matrices['PLS_PCA']

In [ ]:
# Juntar os arquivos de rbo em um único dataframe para synthetic

rbo_synthetic_list = []

# Adicionar dados dos modelos base
for model_name, df in rbo_results['synthetic'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = model_name
        rbo_synthetic_list.append(df_copy)

# Adicionar dados do PCA_aggregator
for model_name, df in rbo_pca_results['synthetic'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = f'{model_name}_PCA'
        rbo_synthetic_list.append(df_copy)

rbo_synthetic = pd.concat(rbo_synthetic_list, ignore_index=True)

print(f"Total de linhas: {len(rbo_synthetic)}")
print(f"Modelos: {sorted(rbo_synthetic['Model'].unique().tolist())}")
rbo_synthetic

In [ ]:
# Criar matrizes de comparação para cada modelo (formato co-ocorrência)
# Method_1 nas linhas, Method_2 nas colunas
# Matriz triangular inferior preenchida (abaixo da diagonal)

print("="*80)
print("CRIANDO MATRIZES DE COMPARAÇÃO POR MODELO (SYNTHETIC)")
print("="*80)

# Dicionário para armazenar as matrizes de cada modelo
synthetic_matrices = {}

# Obter lista de modelos únicos
models_list = sorted(rbo_synthetic['Model'].unique())

print(f"\nModelos encontrados: {models_list}\n")

# Para cada modelo, criar uma matriz pivotada triangular inferior
for model in models_list:
    # Filtrar dados do modelo
    model_data = rbo_synthetic[rbo_synthetic['Model'] == model].copy()
    
    # Identificar a coluna de valores (deve ser numérica e não Method_1/Method_2/Model)
    value_cols = [col for col in model_data.columns 
                  if col not in ['Method_1', 'Method_2', 'Model'] 
                  and pd.api.types.is_numeric_dtype(model_data[col])]
    
    if value_cols:
        value_col = value_cols[0]  # Usar a primeira coluna de valores encontrada
        
        # Reorganizar para manter apenas triangular inferior
        # Se existir (A, B) e (B, A), combinar mantendo apenas onde row >= col
        model_data_processed = []
        
        for _, row in model_data.iterrows():
            m1, m2, val = row['Method_1'], row['Method_2'], row[value_col]
            
            # Sempre colocar o "maior" método na linha (Method_1)
            if m1 < m2:  # Comparação lexicográfica
                m1, m2 = m2, m1  # Trocar
            
            model_data_processed.append({
                'Method_1': m1,
                'Method_2': m2,
                value_col: val
            })
        
        # Criar DataFrame processado
        df_processed = pd.DataFrame(model_data_processed)
        
        # Remover duplicatas (manter apenas uma entrada por par)
        df_processed = df_processed.drop_duplicates(subset=['Method_1', 'Method_2'], keep='first')
        
        # Criar matriz pivot
        matrix = df_processed.pivot(
            index='Method_1',
            columns='Method_2',
            values=value_col
        )
        
        # Garantir que a ordem seja consistente (ordenar índice e colunas)
        matrix = matrix.sort_index(axis=0).sort_index(axis=1)
        
        synthetic_matrices[model] = matrix
        
        print(f"✅ {model.ljust(15)} → Matriz {matrix.shape[0]}x{matrix.shape[1]} ({value_col}) - Triangular inferior")
    else:
        print(f"⚠️  {model.ljust(15)} → Nenhuma coluna de valores encontrada")

print(f"\n{'='*80}")
print(f"✅ Total: {len(synthetic_matrices)} matrizes criadas")
print(f"{'='*80}")

# Exibir exemplo de uma matriz
if synthetic_matrices:
    example_model = models_list[0]
    print(f"\n📊 EXEMPLO - Matriz triangular inferior para {example_model}:")
    print(synthetic_matrices[example_model])
    print("\n💡 Valores abaixo da diagonal principal estão preenchidos, acima estão vazios (NaN)")


In [ ]:
synthetic_matrices['PLS_PCA']

In [ ]:
# Juntar os arquivos de rbo em um único dataframe para milk

rbo_milk_list = []

# Adicionar dados dos modelos base
for model_name, df in rbo_results['milk'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = model_name
        rbo_milk_list.append(df_copy)

# Adicionar dados do PCA_aggregator
for model_name, df in rbo_pca_results['milk'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = f'{model_name}_PCA'
        rbo_milk_list.append(df_copy)

rbo_milk = pd.concat(rbo_milk_list, ignore_index=True)

print(f"Total de linhas: {len(rbo_milk)}")
print(f"Modelos: {sorted(rbo_milk['Model'].unique().tolist())}")
rbo_milk

In [ ]:
# Criar matrizes de comparação para cada modelo (formato co-ocorrência)
# Method_1 nas linhas, Method_2 nas colunas
# Matriz triangular inferior preenchida (abaixo da diagonal)

print("="*80)
print("CRIANDO MATRIZES DE COMPARAÇÃO POR MODELO (MILK)")
print("="*80)

# Dicionário para armazenar as matrizes de cada modelo
milk_matrices = {}

# Obter lista de modelos únicos
models_list = sorted(rbo_milk['Model'].unique())

print(f"\nModelos encontrados: {models_list}\n")

# Para cada modelo, criar uma matriz pivotada triangular inferior
for model in models_list:
    # Filtrar dados do modelo
    model_data = rbo_milk[rbo_milk['Model'] == model].copy()
    
    # Identificar a coluna de valores (deve ser numérica e não Method_1/Method_2/Model)
    value_cols = [col for col in model_data.columns 
                  if col not in ['Method_1', 'Method_2', 'Model'] 
                  and pd.api.types.is_numeric_dtype(model_data[col])]
    
    if value_cols:
        value_col = value_cols[0]  # Usar a primeira coluna de valores encontrada
        
        # Reorganizar para manter apenas triangular inferior
        # Se existir (A, B) e (B, A), combinar mantendo apenas onde row >= col
        model_data_processed = []
        
        for _, row in model_data.iterrows():
            m1, m2, val = row['Method_1'], row['Method_2'], row[value_col]
            
            # Sempre colocar o "maior" método na linha (Method_1)
            if m1 < m2:  # Comparação lexicográfica
                m1, m2 = m2, m1  # Trocar
            
            model_data_processed.append({
                'Method_1': m1,
                'Method_2': m2,
                value_col: val
            })
        
        # Criar DataFrame processado
        df_processed = pd.DataFrame(model_data_processed)
        
        # Remover duplicatas (manter apenas uma entrada por par)
        df_processed = df_processed.drop_duplicates(subset=['Method_1', 'Method_2'], keep='first')
        
        # Criar matriz pivot
        matrix = df_processed.pivot(
            index='Method_1',
            columns='Method_2',
            values=value_col
        )
        
        # Garantir que a ordem seja consistente (ordenar índice e colunas)
        matrix = matrix.sort_index(axis=0).sort_index(axis=1)
        
        milk_matrices[model] = matrix
        
        print(f"✅ {model.ljust(15)} → Matriz {matrix.shape[0]}x{matrix.shape[1]} ({value_col}) - Triangular inferior")
    else:
        print(f"⚠️  {model.ljust(15)} → Nenhuma coluna de valores encontrada")

print(f"\n{'='*80}")
print(f"✅ Total: {len(milk_matrices)} matrizes criadas")
print(f"{'='*80}")

# Exibir exemplo de uma matriz
if milk_matrices:
    example_model = models_list[0]
    print(f"\n📊 EXEMPLO - Matriz triangular inferior para {example_model}:")
    print(milk_matrices[example_model])
    print("\n💡 Valores abaixo da diagonal principal estão preenchidos, acima estão vazios (NaN)")


In [ ]:
milk_matrices['PLS']

In [ ]:
# Juntar os arquivos de rbo em um único dataframe para bank_notes

rbo_bank_notes_list = []

# Adicionar dados dos modelos base
for model_name, df in rbo_results['bank_notes'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = model_name
        rbo_bank_notes_list.append(df_copy)

# Adicionar dados do PCA_aggregator
for model_name, df in rbo_pca_results['bank_notes'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = f'{model_name}_PCA'
        rbo_bank_notes_list.append(df_copy)

rbo_bank_notes = pd.concat(rbo_bank_notes_list, ignore_index=True)

print(f"Total de linhas: {len(rbo_bank_notes)}")
print(f"Modelos: {sorted(rbo_bank_notes['Model'].unique().tolist())}")
rbo_bank_notes

In [ ]:
# Criar matrizes de comparação para cada modelo (formato co-ocorrência)
# Method_1 nas linhas, Method_2 nas colunas
# Matriz triangular inferior preenchida (abaixo da diagonal)

print("="*80)
print("CRIANDO MATRIZES DE COMPARAÇÃO POR MODELO (BANK_NOTES)")
print("="*80)

# Dicionário para armazenar as matrizes de cada modelo
bank_notes_matrices = {}

# Obter lista de modelos únicos
models_list = sorted(rbo_bank_notes['Model'].unique())

print(f"\nModelos encontrados: {models_list}\n")

# Para cada modelo, criar uma matriz pivotada triangular inferior
for model in models_list:
    # Filtrar dados do modelo
    model_data = rbo_bank_notes[rbo_bank_notes['Model'] == model].copy()
    
    # Identificar a coluna de valores (deve ser numérica e não Method_1/Method_2/Model)
    value_cols = [col for col in model_data.columns 
                  if col not in ['Method_1', 'Method_2', 'Model'] 
                  and pd.api.types.is_numeric_dtype(model_data[col])]
    
    if value_cols:
        value_col = value_cols[0]  # Usar a primeira coluna de valores encontrada
        
        # Reorganizar para manter apenas triangular inferior
        # Se existir (A, B) e (B, A), combinar mantendo apenas onde row >= col
        model_data_processed = []
        
        for _, row in model_data.iterrows():
            m1, m2, val = row['Method_1'], row['Method_2'], row[value_col]
            
            # Sempre colocar o "maior" método na linha (Method_1)
            if m1 < m2:  # Comparação lexicográfica
                m1, m2 = m2, m1  # Trocar
            
            model_data_processed.append({
                'Method_1': m1,
                'Method_2': m2,
                value_col: val
            })
        
        # Criar DataFrame processado
        df_processed = pd.DataFrame(model_data_processed)
        
        # Remover duplicatas (manter apenas uma entrada por par)
        df_processed = df_processed.drop_duplicates(subset=['Method_1', 'Method_2'], keep='first')
        
        # Criar matriz pivot
        matrix = df_processed.pivot(
            index='Method_1',
            columns='Method_2',
            values=value_col
        )
        
        # Garantir que a ordem seja consistente (ordenar índice e colunas)
        matrix = matrix.sort_index(axis=0).sort_index(axis=1)
        
        bank_notes_matrices[model] = matrix
        
        print(f"✅ {model.ljust(15)} → Matriz {matrix.shape[0]}x{matrix.shape[1]} ({value_col}) - Triangular inferior")
    else:
        print(f"⚠️  {model.ljust(15)} → Nenhuma coluna de valores encontrada")

print(f"\n{'='*80}")
print(f"✅ Total: {len(bank_notes_matrices)} matrizes criadas")
print(f"{'='*80}")

# Exibir exemplo de uma matriz
if bank_notes_matrices:
    example_model = models_list[0]
    print(f"\n📊 EXEMPLO - Matriz triangular inferior para {example_model}:")
    print(bank_notes_matrices[example_model])
    print("\n💡 Valores abaixo da diagonal principal estão preenchidos, acima estão vazios (NaN)")


In [ ]:
bank_notes_matrices['PLS_PCA']

In [ ]:
# Juntar os arquivos de rbo em um único dataframe para soil_types

rbo_soil_types_list = []

# Adicionar dados dos modelos base
for model_name, df in rbo_results['soil_types'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = model_name
        rbo_soil_types_list.append(df_copy)

# Adicionar dados do PCA_aggregator
for model_name, df in rbo_pca_results['soil_types'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = f'{model_name}_PCA'
        rbo_soil_types_list.append(df_copy)

rbo_soil_types = pd.concat(rbo_soil_types_list, ignore_index=True)

print(f"Total de linhas: {len(rbo_soil_types)}")
print(f"Modelos: {sorted(rbo_soil_types['Model'].unique().tolist())}")
rbo_soil_types

In [ ]:
# Criar matrizes de comparação para cada modelo (formato co-ocorrência)
# Method_1 nas linhas, Method_2 nas colunas
# Matriz triangular inferior preenchida (abaixo da diagonal)

print("="*80)
print("CRIANDO MATRIZES DE COMPARAÇÃO POR MODELO (SOIL_TYPES)")
print("="*80)

# Dicionário para armazenar as matrizes de cada modelo
soil_types_matrices = {}

# Obter lista de modelos únicos
models_list = sorted(rbo_soil_types['Model'].unique())

print(f"\nModelos encontrados: {models_list}\n")

# Para cada modelo, criar uma matriz pivotada triangular inferior
for model in models_list:
    # Filtrar dados do modelo
    model_data = rbo_soil_types[rbo_soil_types['Model'] == model].copy()
    
    # Identificar a coluna de valores (deve ser numérica e não Method_1/Method_2/Model)
    value_cols = [col for col in model_data.columns 
                  if col not in ['Method_1', 'Method_2', 'Model'] 
                  and pd.api.types.is_numeric_dtype(model_data[col])]
    
    if value_cols:
        value_col = value_cols[0]  # Usar a primeira coluna de valores encontrada
        
        # Reorganizar para manter apenas triangular inferior
        # Se existir (A, B) e (B, A), combinar mantendo apenas onde row >= col
        model_data_processed = []
        
        for _, row in model_data.iterrows():
            m1, m2, val = row['Method_1'], row['Method_2'], row[value_col]
            
            # Sempre colocar o "maior" método na linha (Method_1)
            if m1 < m2:  # Comparação lexicográfica
                m1, m2 = m2, m1  # Trocar
            
            model_data_processed.append({
                'Method_1': m1,
                'Method_2': m2,
                value_col: val
            })
        
        # Criar DataFrame processado
        df_processed = pd.DataFrame(model_data_processed)
        
        # Remover duplicatas (manter apenas uma entrada por par)
        df_processed = df_processed.drop_duplicates(subset=['Method_1', 'Method_2'], keep='first')
        
        # Criar matriz pivot
        matrix = df_processed.pivot(
            index='Method_1',
            columns='Method_2',
            values=value_col
        )
        
        # Garantir que a ordem seja consistente (ordenar índice e colunas)
        matrix = matrix.sort_index(axis=0).sort_index(axis=1)
        
        soil_types_matrices[model] = matrix
        
        print(f"✅ {model.ljust(15)} → Matriz {matrix.shape[0]}x{matrix.shape[1]} ({value_col}) - Triangular inferior")
    else:
        print(f"⚠️  {model.ljust(15)} → Nenhuma coluna de valores encontrada")

print(f"\n{'='*80}")
print(f"✅ Total: {len(soil_types_matrices)} matrizes criadas")
print(f"{'='*80}")

# Exibir exemplo de uma matriz
if soil_types_matrices:
    example_model = models_list[0]
    print(f"\n📊 EXEMPLO - Matriz triangular inferior para {example_model}:")
    print(soil_types_matrices[example_model])
    print("\n💡 Valores abaixo da diagonal principal estão preenchidos, acima estão vazios (NaN)")


In [ ]:
soil_types_matrices['MLP_PCA']

In [ ]:
# Juntar os arquivos de rbo em um único dataframe para soil

rbo_soil_list = []

# Adicionar dados dos modelos base
for model_name, df in rbo_results['soil'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = model_name
        rbo_soil_list.append(df_copy)

# Adicionar dados do PCA_aggregator
for model_name, df in rbo_pca_results['soil'].items():
    if df is not None:
        df_copy = df.copy()
        df_copy['Model'] = f'{model_name}_PCA'
        rbo_soil_list.append(df_copy)

rbo_soil = pd.concat(rbo_soil_list, ignore_index=True)

print(f"Total de linhas: {len(rbo_soil)}")
print(f"Modelos: {sorted(rbo_soil['Model'].unique().tolist())}")
rbo_soil

In [ ]:
# Criar matrizes de comparação para cada modelo (formato co-ocorrência)
# Method_1 nas linhas, Method_2 nas colunas
# Matriz triangular inferior preenchida (abaixo da diagonal)

print("="*80)
print("CRIANDO MATRIZES DE COMPARAÇÃO POR MODELO (SOIL)")
print("="*80)

# Dicionário para armazenar as matrizes de cada modelo
soil_matrices = {}

# Obter lista de modelos únicos
models_list = sorted(rbo_soil['Model'].unique())

print(f"\nModelos encontrados: {models_list}\n")

# Para cada modelo, criar uma matriz pivotada triangular inferior
for model in models_list:
    # Filtrar dados do modelo
    model_data = rbo_soil[rbo_soil['Model'] == model].copy()
    
    # Identificar a coluna de valores (deve ser numérica e não Method_1/Method_2/Model)
    value_cols = [col for col in model_data.columns 
                  if col not in ['Method_1', 'Method_2', 'Model'] 
                  and pd.api.types.is_numeric_dtype(model_data[col])]
    
    if value_cols:
        value_col = value_cols[0]  # Usar a primeira coluna de valores encontrada
        
        # Reorganizar para manter apenas triangular inferior
        # Se existir (A, B) e (B, A), combinar mantendo apenas onde row >= col
        model_data_processed = []
        
        for _, row in model_data.iterrows():
            m1, m2, val = row['Method_1'], row['Method_2'], row[value_col]
            
            # Sempre colocar o "maior" método na linha (Method_1)
            if m1 < m2:  # Comparação lexicográfica
                m1, m2 = m2, m1  # Trocar
            
            model_data_processed.append({
                'Method_1': m1,
                'Method_2': m2,
                value_col: val
            })
        
        # Criar DataFrame processado
        df_processed = pd.DataFrame(model_data_processed)
        
        # Remover duplicatas (manter apenas uma entrada por par)
        df_processed = df_processed.drop_duplicates(subset=['Method_1', 'Method_2'], keep='first')
        
        # Criar matriz pivot
        matrix = df_processed.pivot(
            index='Method_1',
            columns='Method_2',
            values=value_col
        )
        
        # Garantir que a ordem seja consistente (ordenar índice e colunas)
        matrix = matrix.sort_index(axis=0).sort_index(axis=1)
        
        soil_matrices[model] = matrix
        
        print(f"✅ {model.ljust(15)} → Matriz {matrix.shape[0]}x{matrix.shape[1]} ({value_col}) - Triangular inferior")
    else:
        print(f"⚠️  {model.ljust(15)} → Nenhuma coluna de valores encontrada")

print(f"\n{'='*80}")
print(f"✅ Total: {len(soil_matrices)} matrizes criadas")
print(f"{'='*80}")

# Exibir exemplo de uma matriz
if soil_matrices:
    example_model = models_list[0]
    print(f"\n📊 EXEMPLO - Matriz triangular inferior para {example_model}:")
    print(soil_matrices[example_model])
    print("\n💡 Valores abaixo da diagonal principal estão preenchidos, acima estão vazios (NaN)")


In [ ]:
soil_types_matrices['PLS_PCA']

In [ ]:
# Exportar todas as matrizes de comparação para arquivos Excel
# Cada dataset terá um arquivo Excel com uma aba para cada modelo

from pathlib import Path

# Diretório de saída
output_dir = Path.cwd()  # Pasta atual (summary)

# Dicionário com todas as matrizes organizadas por dataset
all_matrices = {
    'forage': forage_matrices,
    'synthetic': synthetic_matrices,
    'milk': milk_matrices,
    'bank_notes': bank_notes_matrices,
    'soil_types': soil_types_matrices,
    'soil': soil_matrices
}

print("="*80)
print("EXPORTANDO MATRIZES DE COMPARAÇÃO PARA ARQUIVOS EXCEL")
print("="*80)
print(f"\nDiretório de saída: {output_dir}\n")

exported_excel_files = []

# Para cada dataset, criar um arquivo Excel
for dataset_name, matrices_dict in all_matrices.items():
    if not matrices_dict:
        print(f"⚠️  {dataset_name.ljust(15)} → Sem matrizes para exportar")
        continue
    
    # Nome do arquivo Excel
    excel_file = output_dir / f'rbo_matrices_{dataset_name}.xlsx'
    
    # Criar um ExcelWriter para escrever múltiplas abas
    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        # Para cada modelo, criar uma aba
        for model_name, matrix_df in matrices_dict.items():
            # Nome da aba (limitado a 31 caracteres no Excel)
            sheet_name = model_name[:31]
            
            # Escrever a matriz na aba
            matrix_df.to_excel(writer, sheet_name=sheet_name)
        
        exported_excel_files.append(excel_file.name)
        
        # Contabilizar abas criadas
        n_sheets = len(matrices_dict)
        print(f"✅ {dataset_name.ljust(15)} → {excel_file.name.ljust(35)} ({n_sheets} abas/modelos)")

print(f"\n{'='*80}")
print(f"✅ Total: {len(exported_excel_files)} arquivos Excel exportados!")
print(f"{'='*80}")

print("\n📁 Arquivos criados:")
for file in exported_excel_files:
    print(f"  - {file}")

print("\n📋 Estrutura dos arquivos:")
print("   • Cada arquivo Excel representa um dataset")
print("   • Cada aba (sheet) representa um modelo diferente")
print("   • Method_1 nas linhas, Method_2 nas colunas")
print("   • Matrizes triangulares inferiores (valores abaixo da diagonal)")

In [ ]:
# Calcular matrizes médias para cada modelo considerando todos os datasets
# Cada modelo terá uma matriz que representa a média dos valores RBO entre todos os datasets

print("="*80)
print("CALCULANDO MATRIZES MÉDIAS POR MODELO (TODOS OS DATASETS)")
print("="*80)

# Dicionário para armazenar as matrizes médias
average_matrices = {}

# Dicionário com todas as matrizes organizadas por dataset (já definido anteriormente)
all_matrices = {
    'forage': forage_matrices,
    'synthetic': synthetic_matrices,
    'milk': milk_matrices,
    'bank_notes': bank_notes_matrices,
    'soil_types': soil_types_matrices,
    'soil': soil_matrices
}

# Identificar todos os modelos únicos presentes em todos os datasets
all_models = set()
for dataset_matrices in all_matrices.values():
    all_models.update(dataset_matrices.keys())

all_models = sorted(all_models)
print(f"\nModelos encontrados: {all_models}\n")

# Para cada modelo, calcular a média das matrizes de todos os datasets
for model in all_models:
    print(f"Processando {model}...")
    
    # Coletar todas as matrizes deste modelo de diferentes datasets
    matrices_to_average = []
    datasets_with_model = []
    
    for dataset_name, dataset_matrices in all_matrices.items():
        if model in dataset_matrices:
            matrices_to_average.append(dataset_matrices[model])
            datasets_with_model.append(dataset_name)
    
    if not matrices_to_average:
        print(f"  ⚠️  {model}: Nenhuma matriz encontrada")
        continue
    
    # Calcular a média das matrizes
    # Usar pd.concat com axis para empilhar e depois calcular a média
    if len(matrices_to_average) == 1:
        # Se há apenas um dataset, a média é a própria matriz
        avg_matrix = matrices_to_average[0].copy()
    else:
        # Empilhar todas as matrizes e calcular a média
        # Usar concat com keys para criar um MultiIndex e depois fazer groupby
        stacked = pd.concat(matrices_to_average, keys=range(len(matrices_to_average)))
        avg_matrix = stacked.groupby(level=1).mean()
    
    average_matrices[model] = avg_matrix
    
    print(f"  ✅ {model.ljust(15)} → Matriz média {avg_matrix.shape[0]}x{avg_matrix.shape[1]} ({len(datasets_with_model)} datasets)")
    print(f"     Datasets: {', '.join(datasets_with_model)}")

print(f"\n{'='*80}")
print(f"✅ Total: {len(average_matrices)} matrizes médias criadas")
print(f"{'='*80}")

# Exibir exemplo de uma matriz média
if average_matrices:
    example_model = all_models[0]
    print(f"\n📊 EXEMPLO - Matriz média para {example_model}:")
    print(average_matrices[example_model])
    print("\n💡 Valores representam a média dos RBOs entre todos os datasets")


In [ ]:
# Exportar matrizes médias para um arquivo Excel
# Um arquivo único com uma aba para cada modelo

from pathlib import Path

# Diretório de saída
output_dir = Path.cwd()  # Pasta atual (summary)

print("="*80)
print("EXPORTANDO MATRIZES MÉDIAS PARA ARQUIVO EXCEL")
print("="*80)
print(f"\nDiretório de saída: {output_dir}\n")

# Nome do arquivo Excel para as médias
excel_file = output_dir / 'rbo_matrices_average_all_datasets.xlsx'

# Criar um ExcelWriter para escrever múltiplas abas
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Para cada modelo, criar uma aba
    for model_name, matrix_df in average_matrices.items():
        # Nome da aba (limitado a 31 caracteres no Excel)
        sheet_name = model_name[:31]
        
        # Escrever a matriz na aba
        matrix_df.to_excel(writer, sheet_name=sheet_name)
    
    # Contabilizar abas criadas
    n_sheets = len(average_matrices)
    print(f"✅ {excel_file.name}")
    print(f"   {n_sheets} abas/modelos criadas")

print(f"\n{'='*80}")
print(f"✅ Arquivo Excel com matrizes médias exportado!")
print(f"{'='*80}")

print("\n📋 Estrutura do arquivo:")
print("   • Arquivo único com médias de todos os datasets")
print("   • Cada aba (sheet) representa um modelo diferente")
print("   • Method_1 nas linhas, Method_2 nas colunas")
print("   • Valores são médias dos RBOs entre todos os datasets")
print("   • Matrizes triangulares inferiores (valores abaixo da diagonal)")


## Calcular Matrizes Médias por Modelo

Calcular a média das matrizes de comparação para cada modelo considerando todos os datasets.

## Exportar Matrizes de Comparação para Excel

Cada dataset terá seu próprio arquivo Excel, com uma aba (sheet) para cada modelo.

## Exportar DataFrames Consolidados para CSV

In [ ]:
# Exportar todos os DataFrames consolidados para arquivos CSV
from pathlib import Path

# Diretório de saída
output_dir = Path.cwd()  # Pasta atual (summary)

# Dicionário com todos os DataFrames
dataframes_to_export = {
    'forage': rbo_forage,
    'synthetic': rbo_synthetic,
    'milk': rbo_milk,
    'bank_notes': rbo_bank_notes,
    'soil_types': rbo_soil_types,
    'soil': rbo_soil
}

print("="*80)
print("EXPORTANDO DATAFRAMES CONSOLIDADOS PARA CSV")
print("="*80)
print(f"\nDiretório de saída: {output_dir}\n")

exported_files = []

for dataset_name, df in dataframes_to_export.items():
    # Nome do arquivo de saída
    output_file = output_dir / f'rbo_consolidated_{dataset_name}.csv'
    
    # Exportar para CSV
    df.to_csv(output_file, index=False, sep=';', encoding='utf-8')
    
    exported_files.append(output_file.name)
    
    print(f"✅ {dataset_name.ljust(15)} → {output_file.name.ljust(35)} ({len(df)} linhas, {len(df.columns)} colunas)")

print(f"\n{'='*80}")
print(f"✅ Total: {len(exported_files)} arquivos exportados com sucesso!")
print(f"{'='*80}")

print("\n📁 Arquivos criados:")
for file in exported_files:
    print(f"  - {file}")

In [ ]:
# Reorganizar os DataFrames para ter comparações com LRC_covariance e LRC_perturbation
# sempre na coluna Method_1

def reorganize_for_lrc_comparison(df):
    """
    Reorganiza o DataFrame para ter:
    1. Primeiro: todas as comparações com LRC_covariance em Method_1
    2. Depois: todas as comparações com LRC_perturbation em Method_1
    """
    
    # Lista para armazenar os DataFrames reorganizados
    reorganized_parts = []
    
    # 1. Processar LRC_covariance
    lrc_cov = df[
        (df['Method_1'] == 'LRC_covariance') | 
        (df['Method_2'] == 'LRC_covariance')
    ].copy()
    
    # Trocar colunas onde LRC_covariance está em Method_2
    mask = lrc_cov['Method_2'] == 'LRC_covariance'
    lrc_cov.loc[mask, ['Method_1', 'Method_2']] = lrc_cov.loc[mask, ['Method_2', 'Method_1']].values
    
    # Ordenar por Model e Method_2
    lrc_cov = lrc_cov.sort_values(['Model', 'Method_2']).reset_index(drop=True)
    reorganized_parts.append(lrc_cov)
    
    # 2. Processar LRC_perturbation
    lrc_pert = df[
        (df['Method_1'] == 'LRC_perturbation') | 
        (df['Method_2'] == 'LRC_perturbation')
    ].copy()
    
    # Trocar colunas onde LRC_perturbation está em Method_2
    mask = lrc_pert['Method_2'] == 'LRC_perturbation'
    lrc_pert.loc[mask, ['Method_1', 'Method_2']] = lrc_pert.loc[mask, ['Method_2', 'Method_1']].values
    
    # Ordenar por Model e Method_2
    lrc_pert = lrc_pert.sort_values(['Model', 'Method_2']).reset_index(drop=True)
    reorganized_parts.append(lrc_pert)
    
    # Concatenar tudo
    reorganized_df = pd.concat(reorganized_parts, ignore_index=True)
    
    return reorganized_df, len(lrc_cov), len(lrc_pert)


# Aplicar reorganização e exportar
print("="*80)
print("EXPORTANDO DATAFRAMES REORGANIZADOS PARA ANÁLISE LRC")
print("="*80)
print(f"\nDiretório de saída: {output_dir}\n")

exported_lrc_files = []

for dataset_name, df in dataframes_to_export.items():
    # Reorganizar o DataFrame
    df_reorganized, n_cov, n_pert = reorganize_for_lrc_comparison(df)
    
    # Nome do arquivo de saída
    output_file = output_dir / f'rbo_lrc_comparison_{dataset_name}.csv'
    
    # Exportar para CSV
    df_reorganized.to_csv(output_file, index=False, sep=';', encoding='utf-8')
    
    exported_lrc_files.append(output_file.name)
    
    print(f"✅ {dataset_name.ljust(15)} → {output_file.name.ljust(40)}")
    print(f"   {''.ljust(18)} ({len(df_reorganized)} linhas: {n_cov} LRC_covariance + {n_pert} LRC_perturbation)")

print(f"\n{'='*80}")
print(f"✅ Total: {len(exported_lrc_files)} arquivos reorganizados exportados!")
print(f"{'='*80}")

print("\n📁 Arquivos criados:")
for file in exported_lrc_files:
    print(f"  - {file}")
    
print("\n📋 Estrutura dos arquivos:")
print("   1️⃣  Comparações com LRC_covariance (Method_1)")
print("   2️⃣  Comparações com LRC_perturbation (Method_1)")

In [ ]:
# Visualizar um exemplo: forage reorganizado
df_example_lrc, n_cov_ex, n_pert_ex = reorganize_for_lrc_comparison(rbo_forage)

print("📊 EXEMPLO DE ESTRUTURA REORGANIZADA (forage):")
print("="*80)

print(f"\n1️⃣  Primeiras linhas - Comparações com LRC_covariance ({n_cov_ex} linhas):")
print(df_example_lrc.head(10))

print(f"\n2️⃣  Linhas após transição - Comparações com LRC_perturbation ({n_pert_ex} linhas):")
print(df_example_lrc.iloc[n_cov_ex:n_cov_ex+10])

print("\n" + "="*80)
print("✅ Estrutura confirmada: LRC_covariance sempre em Method_1 (primeira metade),")
print("                        LRC_perturbation sempre em Method_1 (segunda metade)")